<a href="https://colab.research.google.com/github/liamscanlon5/DS_Analytics_Demand_Estimation_FinalProject/blob/main/Notebook_2_DataScientist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 2. Demand Estimation

We will estimate a logit-style demand model using linear regression. The model is:

$$
\log(s_{bt}) = \alpha_0 + \alpha_t + \gamma_b + \beta_{price}p_{bt} + \beta_{rating}r_{bt} + \sum_{\ell=1}^L \beta_\ell x_{bt\ell} + \epsilon_{bt}.
$$

Here:

- $b$ indexes brands
- $t$ indexes years
- $s_{bt}$ is `brand_share`
- $p_{bt}$ is `avg_price`
- $r_{bt}$ is `avg_rating`
- $x_{bt\ell}$ are the product characteristics
- $\alpha_t$ are year dummy coefficients
- $\gamma_b$ are brand dummy coefficients
- $\beta_{price}$ is **one constant price coefficient**, shared by all brands and all years

That last point matters: do **not** estimate a different price coefficient for every brand-year. We do not have enough information for that, and it would make the cost calculation impossible to interpret.

Use `pd.get_dummies(..., drop_first=True)` for brand and year dummies. The dropped brand and dropped year become the reference categories, so all dummy coefficients are interpreted relative to those omitted categories.

Questions:

1. What is the estimated price coefficient, $\hat{\beta}_{price}$?
2. Is it negative? Why is that important?
3. Which product features are associated with higher demand?
4. Which brand dummy coefficients are largest? Remember that these are interpreted relative to the dropped brand.
5. Which year dummy coefficients are largest? Remember that these are interpreted relative to the dropped year.
6. What is the model's $R^2$?

This part of the work is the **data scientist** role: turning the cleaned data into a model that can be used for prediction and interpretation.

In [4]:
# Imports
import pandas as pd
from sklearn.linear_model import LinearRegression
from google.colab import files

# Upload the data
uploaded = files.upload()

# Read the data
df = pd.read_csv('air_fryers_clean_brand_year.csv')

# Re-define the feature columns we need for the regression
feature_cols = [
    'compact_share',
    'dual_basket_share',
    'oven_style_share',
    'rotisserie_share',
    'window_share'
]

Saving air_fryers_clean_brand_year.csv to air_fryers_clean_brand_year (1).csv


In [ ]:
# Estimate the logit-style demand equation with year and brand fixed effects.

y = df["log_brand_share"]

X_numeric = df[["avg_price", "avg_rating"] + feature_cols]

X_dummies = pd.get_dummies(
    df[["brand", "year"]].astype({"year": "str"}),
    drop_first=True,
    dtype=float
)

X = pd.concat([X_numeric, X_dummies], axis=1)

ols = LinearRegression()
ols.fit(X, y)

coef_table = pd.DataFrame({
    "variable": X.columns,
    "coefficient": ols.coef_
}).sort_values("coefficient", ascending=False)

price_coef = coef_table.loc[
    coef_table["variable"] == "avg_price",
    "coefficient"
].iloc[0]

r2 = ols.score(X, y)

print(f"price coefficient: {price_coef:.4f}")
print(f"R-squared: {r2:.3f}")

coef_table

price coefficient: -0.0377
R-squared: 0.763


,variable,coefficient
6,window_share,12.880298
2,compact_share,9.815304
8,brand_cuisinart,6.422436
12,brand_ninja,5.838705
11,brand_instant_pot,4.626260
10,brand_gowise usa,3.938996
14,brand_oster,3.928074
13,brand_nuwave,3.544883
7,brand_cosori,2.551946
4,oven_style_share,1.941774



1.   The estimated price coefficient is -0.0377.  
2.   It's negative which makes sense, since there should be lower demand when the price increases.
3. The product features associated with higher demand are `oven_style_share`, `compact_share`, and `window_share`.  
4. The largest brand dummy coefficients are Cuisnart, Ninja, Instant Post, GoWise, and Oster.
5. The year dummy coefficients are small, 2020 is the largest, 2021 is the second largest.
6. The R-squared is 0.763
